# T004 — Análise exploratória reproduzível

Notebook de execução local. Ele chama as funções de `src/evaluation/exploratoria.py`; a lógica não está duplicada aqui.

Pré-requisito: tabelas Parquet geradas pela T003 em `data/processed`. O controlador executa este notebook sobre o perfil OpenFootball aprovado, com 3.040 partidas de 2018–2025. Os outputs serão produzidos pelo controlador ao executar o notebook; nenhum resultado está registrado neste arquivo.

In [ ]:
from pathlib import Path

from evaluation.exploratoria import carregar_base, exportar_analise, resumo_exploratorio

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PASTA_PROCESSED = RAIZ / "data" / "processed"
PASTA_SAIDA = RAIZ / "reports" / "exploratoria"
# A origem só é declarada real quando o manifesto local está presente.
manifesto = RAIZ / "data/raw/openfootball_2018_2025/manifest.json"
ORIGEM_DOS_DADOS = "OpenFootball real (manifesto local)" if manifesto.is_file() else "não informada"

## Carregamento da base

Cada linha é uma partida. Os joins com as dimensões são validados (many_to_one); chaves sem correspondência interrompem a execução.

In [ ]:
base = carregar_base(PASTA_PROCESSED)
print(
    f"Origem: {ORIGEM_DOS_DADOS} | partidas: {len(base)} | "
    f"temporadas: {sorted(base['temporada'].unique())}"
)
base.head()

## Resumo exploratório

Médias de gols, distribuição H/D/A, agregados por temporada e equipe, e indicadores de qualidade. Nada é removido automaticamente.

In [ ]:
resumo = resumo_exploratorio(base)
print(resumo["media_gols"])
print(resumo["distribuicao_resultados"])
print(resumo["qualidade"])

## Exportação de relatório e gráficos

Grava `resumo.json` e os PNGs de gols por temporada, distribuição H/D/A e pontos por equipe.

In [ ]:
caminhos = exportar_analise(base, PASTA_SAIDA)
for chave, caminho in caminhos.items():
    print(chave, caminho.relative_to(RAIZ))

## Limites

- Análise descritiva. Diferenças entre equipes, temporadas ou mando não são causais.
- Período real validado pelo controlador: 2018–2025. A temporada 2026 é reservada e não é lida.
- Dados sintéticos servem apenas para testes; a flag `ORIGEM_DOS_DADOS` deve refletir a origem real.
- Hipóteses H01–H05 (PRD 2.3) são planos de investigação; não estão confirmadas nesta etapa.